In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

In [4]:
# Load the data
file_path = '../data/chronic_disease_indicators.csv'
data = pd.read_csv(file_path)

# Drop columns with all missing values
df = data.dropna(axis=1, how='all')

# Drop columns with redundant or not useful information (footnote details)
columns_to_drop = ['DataValueFootnoteSymbol', 'DataValueFootnote']
df = df.drop(columns=columns_to_drop)

In [6]:
# Count the number of records for each question
question_counts = df['Question'].value_counts()

# Display the counts
print(question_counts)

df = df[df['Question'] == "Diseases of the heart mortality among all people, underlying cause"]

Question
Binge drinking frequency among adults who binge drink                            5720
Binge drinking intensity among adults who binge drink                            5680
Diseases of the heart mortality among all people, underlying cause               5616
Cerebrovascular disease (stroke) mortality among all people, underlying cause    5616
Asthma mortality among all people, underlying cause                              5616
                                                                                 ... 
Infants who were exclusively breastfed through 6 months                           122
Infants who were breastfed at 12 months                                           122
Incidence of treated end-stage kidney disease                                     104
No broadband internet subscription among households                               104
Food insecure in the past 12 months among households                               55
Name: count, Length: 109, dtype: int64


In [8]:
df.to_csv('cleaned_data.csv',index=False)

Filter: Diseases of the heart mortality among all people, underlying cause, 
DataValueUnit cases per 100,000

In [10]:
print(df.columns)

Index(['YearStart', 'YearEnd', 'LocationAbbr', 'LocationDesc', 'DataSource',
       'Topic', 'Question', 'DataValueUnit', 'DataValueType', 'DataValue',
       'DataValueAlt', 'LowConfidenceLimit', 'HighConfidenceLimit',
       'StratificationCategory1', 'Stratification1', 'Geolocation',
       'LocationID', 'TopicID', 'QuestionID', 'DataValueTypeID',
       'StratificationCategoryID1', 'StratificationID1'],
      dtype='object')


In [12]:
columns_to_drop = ['YearEnd', 'LocationDesc','DataSource',
       'Topic','DataValueAlt', 'LowConfidenceLimit', 'HighConfidenceLimit','Geolocation',
       'LocationID', 'TopicID', 'QuestionID', 'DataValueTypeID',
       'StratificationCategoryID1', 'StratificationID1']
df = df.drop(columns=columns_to_drop)


In [14]:
df.head()

,YearStart,LocationAbbr,Question,DataValueUnit,DataValueType,DataValue,StratificationCategory1,Stratification1
13549,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Hawaiian or Pacific Islander, non-Hispanic"
13733,2019,AK,Diseases of the heart mortality among all peop...,Number,Number,28.0,Race/Ethnicity,"Multiracial, non-Hispanic"
13800,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,97.2,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
14016,2019,AR,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Multiracial, non-Hispanic"
14052,2019,AL,Diseases of the heart mortality among all peop...,"cases per 100,000",Age-adjusted Rate,62.5,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"


In [18]:
print(df['DataValueUnit'].value_counts())


DataValueUnit
cases per 100,000    3588
Number               2028
Name: count, dtype: int64


In [22]:
df=df[df['DataValueUnit']=='cases per 100,000']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3588 entries, 13549 to 276748
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   YearStart                3588 non-null   int64  
 1   LocationAbbr             3588 non-null   object 
 2   Question                 3588 non-null   object 
 3   DataValueUnit            3588 non-null   object 
 4   DataValueType            3588 non-null   object 
 5   DataValue                2891 non-null   float64
 6   StratificationCategory1  3588 non-null   object 
 7   Stratification1          3588 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 252.3+ KB


In [36]:
df.head()

,YearStart,LocationAbbr,Question,DataValueUnit,DataValueType,DataValue,StratificationCategory1,Stratification1
13549,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Hawaiian or Pacific Islander, non-Hispanic"
13800,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,97.2,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
14016,2019,AR,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Multiracial, non-Hispanic"
14052,2019,AL,Diseases of the heart mortality among all peop...,"cases per 100,000",Age-adjusted Rate,62.5,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
14070,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,148.6,Sex,Female


In [24]:
print(df['DataValueType'].value_counts())

DataValueType
Crude Rate           2028
Age-adjusted Rate    1560
Name: count, dtype: int64


In [30]:
df=df[df['DataValueType']=='Crude Rate']
df.head()

,YearStart,LocationAbbr,Question,DataValueUnit,DataValueType,DataValue,StratificationCategory1,Stratification1
13549,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Hawaiian or Pacific Islander, non-Hispanic"
13800,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,97.2,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
14016,2019,AR,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,NaN,Race/Ethnicity,"Multiracial, non-Hispanic"
14070,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,148.6,Sex,Female
14073,2019,AK,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,115.2,Overall,Overall


In [43]:
df=df.dropna(subset=['DataValue'])
df.shape[0]
df.head()

,YearStart,LocationAbbr,Question,DataValueUnit,DataValueType,DataValue,StratificationCategory1,Stratification1
13800,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,97.2,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
14070,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,148.6,Sex,Female
14073,2019,AK,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,115.2,Overall,Overall
14928,2019,AK,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,623.4,Age,Age >=65
15032,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,796.1,Age,Age >=65


In [45]:
print(df['StratificationCategory1'].value_counts())

StratificationCategory1
Race/Ethnicity    744
Age               467
Sex               312
Overall           156
Name: count, dtype: int64


In [49]:
df=df[df['StratificationCategory1']=='Race/Ethnicity']
df.head()

,YearStart,LocationAbbr,Question,DataValueUnit,DataValueType,DataValue,StratificationCategory1,Stratification1
13800,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,97.2,Race/Ethnicity,"American Indian or Alaska Native, non-Hispanic"
15183,2019,AK,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,132.3,Race/Ethnicity,"Black, non-Hispanic"
15391,2019,AR,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,230.9,Race/Ethnicity,"Black, non-Hispanic"
15976,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,140.3,Race/Ethnicity,"Black, non-Hispanic"
16196,2019,AZ,Diseases of the heart mortality among all peop...,"cases per 100,000",Crude Rate,65.4,Race/Ethnicity,"Asian, non-Hispanic"


In [51]:
columns_to_drop=['Question','DataValueUnit','DataValueType','StratificationCategory1']
df=df.drop(columns=columns_to_drop)
df.head()

,YearStart,LocationAbbr,DataValue,Stratification1
13800,2019,AZ,97.2,"American Indian or Alaska Native, non-Hispanic"
15183,2019,AK,132.3,"Black, non-Hispanic"
15391,2019,AR,230.9,"Black, non-Hispanic"
15976,2019,AZ,140.3,"Black, non-Hispanic"
16196,2019,AZ,65.4,"Asian, non-Hispanic"


Group by: YearStart, LocationAbbr
Response: DataValue

In [53]:
df.shape

(744, 4)

In [59]:
df.to_csv('../data/final_training_data.csv',index=False)